# Recommender Pipeline and Model Development

This notebook is the canonical grader walkthrough. It shows the full project story:

1. **Exploration in notebooks** - `Data_Extraction.ipynb` and `Cleaned Up Notebook.ipynb` were used to figure out the data, sampling, feature engineering, baselines, and collaborative filtering.
2. **Migration into clean modules** - the working notebook code was moved into `src/data/`, `src/models/`, `src/evaluation.py`, and `src/pipeline.py` so the project can be reproduced from scripts.
3. **Final reproducible run** - this notebook imports those modules, runs the full recommender pipeline, and displays metrics/figures.

The old notebooks are intentionally kept as an audit trail. This notebook is cleaner because it demonstrates the final implementation instead of duplicating every function inline.


## 1. Repo Setup

`DATA_MODE = "synthetic"` works locally without the 189 GB metadata archive. If `data/influencers.txt` exists, synthetic posts use real influencer profiles; otherwise the code uses built-in demo profiles so a fresh clone still runs.

For a real-data Colab run, set `DATA_MODE = "real_metadata"` and point `EXTRACTED_METADATA_DIR` to the extracted `.info` / JSON metadata sample.


In [ ]:
from pathlib import Path
import sys

# Pick one mode: "synthetic" | "parquet" | "real_metadata"
DATA_MODE = "synthetic"

# Quick demo: 1000. Main project result: 10000. Larger benchmark: 20000/50000/100000.
TARGET_POSTS = 1000
SYNTHETIC_INFLUENCERS = 120
TOP_K = 5
RANDOM_SEED = 42

EXTRACTED_METADATA_DIR = None  # e.g. Path("/content/Post_metadata_10000_extracted")
POSTS_PARQUET = None           # e.g. Path("/content/drive/MyDrive/dsci351_artifacts/posts_base_10000.parquet")


def find_repo_root() -> Path:
    candidates = [Path.cwd(), *Path.cwd().parents]
    try:
        candidates.insert(0, Path(__file__).resolve().parent.parent)
    except NameError:
        pass

    colab_repo = Path("/content/drive/MyDrive/DSCI351/Class Project/DSCI_Recommender_System_Class_Proj")
    if colab_repo.exists():
        candidates.insert(0, colab_repo)

    for candidate in candidates:
        if (candidate / "src" / "pipeline.py").exists():
            return candidate

    raise FileNotFoundError("Open this notebook from inside the project repo.")


REPO = find_repo_root()
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

DATA_DIR = REPO / "data"
OUTPUT_DIR = REPO / "artifacts"

USE_SYNTHETIC = DATA_MODE == "synthetic"
USE_PARQUET = DATA_MODE == "parquet" and POSTS_PARQUET is not None
USE_EXTRACTED = DATA_MODE == "real_metadata" and EXTRACTED_METADATA_DIR is not None

print("Repo:", REPO)
print("Data mode:", DATA_MODE)
print("Target posts:", TARGET_POSTS)
print("Influencers file exists:", (DATA_DIR / "influencers.txt").exists())


## 2. Development Audit Trail

The project was not written as final modules first. The workflow was:

| Development notebook | What it proved | Final module/script |
|---|---|---|
| `Data_Extraction.ipynb` | Drive setup, dataset discovery, metadata archive extraction, mapping inspection | `docs/Colab_Setup.md`, `scripts/build_training_subset.py`, `src/data/preprocess.py` |
| `Cleaned Up Notebook.ipynb` | Parsing, strategy labels, engagement features, global/category baselines, user-based CF prototype | `src/data/preprocess.py`, `src/models/baselines.py`, `src/models/collaborative.py` |
| This notebook | Final clean run, content-based model, hybrid model, time split, ranking metrics | `src/models/content_based.py`, `src/models/hybrid.py`, `src/evaluation.py`, `src/pipeline.py` |

The cells below inspect the final module functions so the notebook still shows the actual code used for grading.


In [ ]:
import inspect
from src.data import preprocess
from src.models import baselines, collaborative, content_based, hybrid
from src import evaluation

objects_to_show = [
    ("Strategy label", preprocess.build_strategy_label),
    ("Engagement features", preprocess.add_engagement_features),
    ("Global baseline scores", baselines.build_strategy_scores),
    ("User-based CF", collaborative.recommend_user_based_cf),
    ("Content-based recommender", content_based.ContentBasedRecommender.fit),
    ("Hybrid recommender", hybrid.recommend_hybrid),
    ("Time split", evaluation.time_based_split),
    ("All-model evaluation", evaluation.evaluate_all_models),
]

for title, obj in objects_to_show:
    print("\n" + "=" * 90)
    print(title)
    print("=" * 90)
    print(inspect.getsource(obj)[:2500])


## 3. Load or Generate Posts

This stage corresponds to the extraction/preprocessing work originally developed in the notebooks. The final pipeline supports three inputs:

- Synthetic posts for local grading/reproduction
- Extracted real `.info` metadata from Colab
- A saved `posts_base_*.parquet` from a prior run


In [ ]:
from src.pipeline import PipelineConfig, load_posts_base, resolve_output_dir
from src.data.preprocess import assign_pseudo_ratings

config = PipelineConfig(
    data_dir=DATA_DIR,
    output_dir=OUTPUT_DIR,
    target_posts=TARGET_POSTS,
    posts_parquet=POSTS_PARQUET if USE_PARQUET else None,
    extracted_metadata_dir=EXTRACTED_METADATA_DIR if USE_EXTRACTED else None,
    synthetic=USE_SYNTHETIC,
    synthetic_influencers=SYNTHETIC_INFLUENCERS,
    k=TOP_K,
    seed=RANDOM_SEED,
)

run_output_dir = resolve_output_dir(config)
posts_base_df = load_posts_base(config, run_output_dir)
rated_preview = assign_pseudo_ratings(posts_base_df)

print(f"Posts: {len(posts_base_df):,}")
print(f"Influencers: {posts_base_df['influencer_name'].nunique():,}")
print(f"Strategies: {posts_base_df['strategy'].nunique():,}")
print("Columns:", list(posts_base_df.columns))

posts_base_df[[
    "influencer_name", "category", "followers", "likes", "comments",
    "time_bucket", "caption_bucket", "hashtag_bucket", "ad_bucket", "media_bucket",
    "strategy", "engagement_rate", "log_engagement_score"
]].head()


## 4. Feature Engineering Checks

These checks make the strategy-item definition visible. Each Instagram post becomes one recommender-system item like:

`evening + medium_caption + few_hashtags + not_ad + image`


In [ ]:
feature_cols = ["time_bucket", "caption_bucket", "hashtag_bucket", "ad_bucket", "media_bucket"]

print("Bucket counts")
for col in feature_cols:
    print("\n", col)
    print(posts_base_df[col].value_counts().head(10).to_string())

print("\nTop strategies")
print(posts_base_df["strategy"].value_counts().head(12).to_string())

print("\nPseudo-rating distribution")
print(rated_preview["pseudo_rating"].value_counts(dropna=False).sort_index().to_string())


## 5. Train/Test Split and Model Artifacts

This is where the exploratory recommender logic became a formal experiment:

- Sort each influencer's posts by time
- Train on older posts
- Test on the most recent 20%
- Build artifacts for all five recommenders


In [ ]:
from src.evaluation import prepare_evaluation_frames, build_model_artifacts

train_df, test_df = prepare_evaluation_frames(posts_base_df)
artifacts = build_model_artifacts(train_df)

print(f"Train posts: {len(train_df):,}")
print(f"Test posts: {len(test_df):,}")
print(f"Train influencers: {train_df['influencer_name'].nunique():,}")
print(f"Test influencers: {test_df['influencer_name'].nunique():,}")
print("Interaction matrix shape:", artifacts["interaction_matrix"].shape)
print("User similarity shape:", artifacts["user_similarity_df"].shape)
print("Content strategy vectors shape:", artifacts["content_recommender"].strategy_vectors_.shape)

train_df[["influencer_name", "datetime", "strategy", "pseudo_rating"]].head()


## 6. Example Recommendations from Each Model

The recommendation output is a ranked list of content strategies, not individual posts or captions. This cell shows the exact output shape for one influencer.


In [ ]:
from src.models.baselines import recommend_global_for_influencer, recommend_category_for_influencer
from src.models.collaborative import recommend_user_based_cf
from src.models.hybrid import recommend_hybrid

example_user = test_df["influencer_name"].iloc[0]
print("Example influencer:", example_user)
print("Category:", train_df.loc[train_df["influencer_name"] == example_user, "category"].iloc[0])

recommendations = {
    "global_baseline": recommend_global_for_influencer(example_user, train_df, artifacts["strategy_scores"], k=TOP_K),
    "category_baseline": recommend_category_for_influencer(example_user, train_df, artifacts["category_strategy_scores"], k=TOP_K),
    "user_based_cf": recommend_user_based_cf(example_user, artifacts["interaction_matrix"], artifacts["user_similarity_df"], k=TOP_K),
    "content_based": artifacts["content_recommender"].recommend(example_user, k=TOP_K),
    "hybrid": recommend_hybrid(
        example_user,
        artifacts["interaction_matrix"],
        artifacts["user_similarity_df"],
        artifacts["content_recommender"],
        k=TOP_K,
        alpha=0.3,
    ),
}

for model, recs in recommendations.items():
    print(f"\n{model}")
    for rank, rec in enumerate(recs, 1):
        print(f"  {rank}. {rec}")


## 7. Evaluate All Models

The final project uses ranking metrics because the task is top-N recommendation:

- Precision@5
- Recall@5
- NDCG@5
- Hit Rate@5

A held-out strategy is relevant when its pseudo-rating is 5, meaning it was in the influencer's top engagement quintile.


In [ ]:
from src.evaluation import evaluate_all_models

results_df, hybrid_alpha = evaluate_all_models(train_df, test_df, k=TOP_K)
print("Tuned hybrid alpha:", hybrid_alpha)
results_df.sort_values("ndcg_at_k", ascending=False)


## 8. Run the Final End-to-End Pipeline

The cells above showed the individual pieces. This final cell runs the production orchestration that writes the same artifacts a professor would see from `python scripts/run_pipeline.py`.


In [ ]:
from src.pipeline import run_pipeline

outputs = run_pipeline(config)
print(f"Run directory: {outputs['run_output_dir']}")
print(f"Results CSV: {outputs['results_path']}")
print(f"Run summary: {outputs['run_summary_path']}")
print(f"Runtime: {outputs['runtime_seconds']:.1f}s")
outputs["results_df"].sort_values("ndcg_at_k", ascending=False)


## 9. Figures

The final pipeline writes metrics and EDA figures under `artifacts/runs/n{TARGET_POSTS}/`.


In [ ]:
try:
    from IPython.display import Image, display
    print("Hybrid alpha:", outputs["hybrid_alpha"])
    for figure_path in outputs["figure_paths"]:
        print(figure_path)
        display(Image(filename=str(figure_path)))
except ImportError:
    print("Hybrid alpha:", outputs["hybrid_alpha"])
    for figure_path in outputs["figure_paths"]:
        print("Figure:", figure_path)


## 10. What This Notebook Proves

- The exploratory notebook work was preserved in `Data_Extraction.ipynb` and `Cleaned Up Notebook.ipynb`.
- The final implementation was migrated into clean, reusable modules.
- The notebook can still show the actual source code used for strategy labels, engagement scoring, baselines, CF, content-based recommendation, hybrid scoring, splitting, and evaluation.
- The same code path writes the committed CSV/PNG artifacts used in the README and final presentation.
